In [1]:
import pathlib
import rdflib
import requests

def pull_literal(e, l, p, graph):

    ''' Pull literal from an entity, by language. '''

    labels = [c for a, b, c in graph.triples((e, p, None))]
    if l != 'skip':
        labels = [c for c in labels if c.language == l]
    if len(labels) == 1:
        return labels[0]

def render_literals(e, l):

    ''' Process markdown for header blocks of literals. '''


    string = '---'+'\n'
    string += f'slug: /{pathlib.Path(e).stem}'+'\n'
    string += '---'+'\n'

    string += f'# {pull_literal(e, l, rdflib.RDFS.label, ontology_graph)}'+'\n'

    string += '#### Class'+'\n'
    string += f'FIAFcore:{pathlib.Path(e).stem}'+'\n'

    string += '#### Reference'+'\n'
    string += pull_literal(e, l, rdflib.RDFS.comment, ontology_graph)+'\n'

    string += '#### Definition'+'\n'
    string += pull_literal(e, l, rdflib.SKOS.definition, ontology_graph)+'\n'

    return string
    
def save_markdown(e, l, s, p):

    ''' Save string to appropriate markdown location. '''

    save_path = p / f'{e}.md'
    save_path.parents[0].mkdir(exist_ok=True, parents=True)
    with open(save_path, 'w') as write_markdown:
        write_markdown.write(s)

def write_markdown_page(page_name, page_text):

    ''' Write markdown pages directly. '''

    md_path = pathlib.Path.home() / 'git' / 'FIAFcore-docs' / 'ontology' / page_name
    with open(md_path, 'w') as md_write:
        md_write.write(page_text)

fiafcore = requests.get('https://raw.githubusercontent.com/FIAF/FIAFcore/main/FIAFcore.ttl')
ontology_graph = rdflib.Graph().parse(data=str(fiafcore.text), format='ttl')

print(len(ontology_graph))

958


In [2]:
# generate class documentation.

class_list = [a for a,b,c in ontology_graph.triples((None, rdflib.RDF.type, rdflib.OWL.Class))]

for entity in class_list:
    for language in ['en', 'es', 'fr']:
        markdown_string = render_literals(entity, language)
        if language == 'en':
            save_dir = pathlib.Path.cwd() / 'ontology' / 'Classes'
        else:
            save_dir = pathlib.Path.cwd() / 'i18n' / language / 'docusaurus-plugin-content-docs' / 'current' / 'Classes'
            
        save_markdown(pathlib.Path(entity).stem, language, markdown_string, save_dir)

print(len(class_list), 'classes processed.')

37 classes processed.


In [3]:
# generate object property documentation.

object_property_list = [a for a,b,c in ontology_graph.triples((None, rdflib.RDF.type, rdflib.OWL.ObjectProperty))]

for object_property in object_property_list:
    for language in ['en', 'es', 'fr']:    
        markdown_string = render_literals(object_property, language)
        domain_list = [c for a,b,c in ontology_graph.triples((object_property, rdflib.RDFS.domain, None))]
        if len(domain_list):
            domain_list = [{'link':str(x), 'label':str(pull_literal(x, language, rdflib.RDFS.label, ontology_graph))} for x in domain_list]
            domain_list = [f"[{y['label']}]({y['link']})" for y in domain_list]
            markdown_string += '\n'+'#### Domain'+'\n' + ', '.join([y for y in domain_list])

        range_list = [c for a,b,c in ontology_graph.triples((object_property, rdflib.RDFS.range, None))]
        if len(range_list):
            range_list = [{'link':str(x), 'label':str(pull_literal(x, language, rdflib.RDFS.label, ontology_graph))} for x in range_list]
            range_list = [f"[{y['label']}]({y['link']})" for y in range_list]
            markdown_string += '\n'+'#### Range'+'\n' + ', '.join([y for y in range_list])

        if language == 'en':
            save_dir = pathlib.Path.cwd() / 'ontology' / 'Properties'
        else:
            save_dir = pathlib.Path.cwd() / 'i18n' / language / 'docusaurus-plugin-content-docs' / 'current' / 'Properties'
            
        save_markdown(pathlib.Path(object_property).stem, language, markdown_string, save_dir)

print(len(object_property_list), 'object properties.')

36 object properties.


In [7]:
# generate datatype property documentation.

datatype_property_list = [a for a,b,c in ontology_graph.triples((None, rdflib.RDF.type, rdflib.OWL.DatatypeProperty))]
for datatype_property in datatype_property_list:
    for language in ['en', 'es', 'fr']:
        markdown_string = render_literals(datatype_property, language)
        domain_list = [c for a,b,c in ontology_graph.triples((datatype_property, rdflib.RDFS.domain, None))]
        if len(domain_list):
            domain_list = [{'link':str(x), 'label':str(pull_literal(x, language, rdflib.RDFS.label, ontology_graph))} for x in domain_list]
            domain_list = [f"[{y['label']}]({y['link']})" for y in domain_list]
            markdown_string += '\n'+'#### Domain'+'\n' + ', '.join([y for y in domain_list])

        range_list = [c for a,b,c in ontology_graph.triples((datatype_property, rdflib.RDFS.range, None))]
        if len(range_list):
            range_list = [{'link':str(x), 'label':str(x).split('#')[-1]} for x in range_list]
            range_list = [f"[{y['label']}]({y['link']})" for y in range_list]
            markdown_string += '\n'+'#### Range'+'\n' + ', '.join([y for y in range_list])

        if language == 'en':
            save_dir = pathlib.Path.cwd() / 'ontology' / 'Properties'
        else:
            save_dir = pathlib.Path.cwd() / 'i18n' / language / 'docusaurus-plugin-content-docs' / 'current' / 'Properties'
            
        save_markdown(pathlib.Path(datatype_property).stem, language, markdown_string, save_dir)

print(len(datatype_property_list), 'datatype properties.')

11 datatype properties.


In [5]:
# write info markdown pages (need es and fr here).

write_markdown_page('Introduction.md', """
# Introduction

FIAFcore is an ontology for film archives.""")

write_markdown_page('About.md', """
# About

Notes about history of the project and how to get involved.""")

In [6]:
# generate sidebar

property_list = sorted(object_property_list+datatype_property_list)
property_list = [pathlib.Path(str(x)).stem for x in property_list]
property_list = [f'"Properties/{x}"' for x in property_list]

class_list = [pathlib.Path(str(x)).stem for x in class_list]
class_list = [f'"Classes/{x}"' for x in class_list]

sidetext = '''
module.exports = {
  docs: [
    {'type':'doc', 'id':'Introduction'},
    {
      type: 'category',
      label: 'Classes',
      collapsible: true,
      collapsed: true,
      items: ['''+','.join(class_list)+''']
    },
    {
      type: 'category',
      label: 'Properties',
      collapsible: true,
      collapsed: true,
      items: ['''+','.join(property_list)+''']
    },
    {'type':'doc', 'id':'About'},
  ]
};
'''

with open(pathlib.Path.cwd() / 'sidebars.js', 'w') as sidebar:
  sidebar.write(sidetext)

print('all done.')

all done.
